In [32]:
import pandas as pd
import numpy as np
import joblib
import re
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [33]:
print("=" * 60)
print("  PELATIHAN MODEL NAÏVE BAYES - DETEKSI PESAN BERBAHAYA ")
print("=" * 60)

  PELATIHAN MODEL NAÏVE BAYES - DETEKSI PESAN BERBAHAYA 


In [34]:
print("\n[1] Memuat dataset...")
 
df = pd.read_csv('Dataset/dataset_fixed.csv')
 
print(f"    Total data   : {len(df)} baris")
print(f"    Kolom        : {list(df.columns)}")
print(f"    Distribusi label:")
for label, count in df['label'].value_counts().items():
    print(f"      - {label}: {count} data")
print(f"    Distribusi kategori:")
for kat, count in df['kategori'].value_counts().items():
    print(f"      - {kat}: {count} data")


[1] Memuat dataset...
    Total data   : 1235 baris
    Kolom        : ['id', 'pesan', 'kategori', 'label']
    Distribusi label:
      - Berisiko: 800 data
      - Tidak Berisiko: 435 data
    Distribusi kategori:
      - Kata Kasar & Ancaman: 400 data
      - Transaksi Aman: 183 data
      - Phishing: 120 data
      - Percakapan Biasa: 96 data
      - Jadwal & Kegiatan: 86 data
      - Social Engineering: 85 data
      - Penipuan: 78 data
      - Informasi Umum: 70 data
      - Pencurian Akun: 66 data
      - Malware: 51 data


In [35]:
stemmer  = StemmerFactory().create_stemmer()
sw_base  = StopWordRemoverFactory().get_stop_words()
sw_extra = [
    'anda','kamu','saya','kami','kita','nya','ini','itu',
    'dengan','untuk','ada','akan','sudah','telah',
    'ya','yg','jg','gak','ga','deh','dong','nih','lah','sih',
    'ku','mu','klo','tapi','jadi','bisa','agar','juga',
]

In [36]:
KATA_KASAR = {
    # Alat kelamin & seksual
    'kontol','memek','pepek','titit','toket','ngentot','entot','ngewe',
    'colmek','coli','masturbasi','bokep','porno','telanjang','bugil',
    'binal','mesum','cabul','bejat',
    # Makian umum
    'anjing','bangsat','bajingan','brengsek','keparat','sialan',
    'babi','goblok','tolol','dungu','geblek','kampret',
    'asu','jancok','jancuk','cuk','kon','taik','tai','setan',
    'iblis','laknat','terkutuk','jahanam','lonte','sundal',
    'pelacur','jalang','murahan','perek',
    # Ancaman kekerasan verbal
    'kubunuh','mampusin','bacok','hajar','tonjok','siksa',
    'aniaya','habisi','musnahkan','gebuk','cekik',
    # Penghinaan SARA
    'kafir','rasis',
}
 
STOPWORDS = set(sw_base + sw_extra) - KATA_KASAR

In [37]:
POLA_URL = (
    r'(bit\.ly|s\.id|rb\.gy|t\.ly|cutt\.ly|tinyurl\.com'
    r'|shorturl\.at|bit\.do|ow\.ly|is\.gd|tiny\.cc'
    r'|[\w-]+\.xyz|[\w-]+\.site)\S*'
)
 
print(f"\n    Stopwords    : {len(STOPWORDS)} kata")
print(f"    Kata kasar   : {len(KATA_KASAR)} kata (blacklist)")


    Stopwords    : 138 kata
    Kata kasar   : 62 kata (blacklist)


In [38]:
print("\n[2] Case Folding...")
print("    Mengubah semua teks menjadi huruf kecil (lowercase)")
 
df['step_casefolding'] = df['pesan'].str.lower().str.strip()
 
print(f"\n    Contoh:")
print(f"    Sebelum : {df['pesan'].iloc[0]}")
print(f"    Sesudah : {df['step_casefolding'].iloc[0]}")


[2] Case Folding...
    Mengubah semua teks menjadi huruf kecil (lowercase)

    Contoh:
    Sebelum : dasar sialan
    Sesudah : dasar sialan


In [39]:
print("\n[3] Tokenisasi...")
print("    Menandai pola khusus lalu memecah teks menjadi token kata")
 
def tokenisasi(text):
    # Tandai URL mencurigakan
    text = re.sub(POLA_URL, 'URL_CURIGA', text)
    # Tandai nomor HP
    text = re.sub(r'\b0\d[\d\-]{8,12}\b', 'NOMOR_HP_ASING', text)
    # Tandai kode OTP (5-8 digit)
    text = re.sub(r'\b\d{5,8}\b', 'KODE_OTP', text)
    # Tandai nominal uang
    text = re.sub(r'rp[\s]?\d+[\.,]?\d*\s*(juta|ribu|rb)?', 'NOMINAL_UANG', text)
    # Hapus sisa angka dan karakter khusus
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'[^a-z_\s]', ' ', text)
    return [t for t in text.split() if len(t) > 0]
 
df['step_tokenisasi'] = df['step_casefolding'].apply(tokenisasi)
 
print(f"\n    Contoh:")
print(f"    Sebelum : {df['step_casefolding'].iloc[0]}")
print(f"    Sesudah : {df['step_tokenisasi'].iloc[0]}")


[3] Tokenisasi...
    Menandai pola khusus lalu memecah teks menjadi token kata

    Contoh:
    Sebelum : dasar sialan
    Sesudah : ['dasar', 'sialan']


In [40]:
print("\n[4] Stopword Removal...")
print("    Membuang kata umum tidak bermakna")
print("    Catatan: kata kasar TIDAK dihapus agar tetap jadi fitur model")
 
def hapus_stopword(tokens):
    return [
        t for t in tokens
        if t.isupper()                          # token khusus (URL_CURIGA, dll)
        or t in KATA_KASAR                      # kata kasar dijaga
        or (t not in STOPWORDS and len(t) > 1)  # kata biasa
    ]
 
df['step_stopword'] = df['step_tokenisasi'].apply(hapus_stopword)
 
print(f"\n    Contoh:")
print(f"    Sebelum : {df['step_tokenisasi'].iloc[0]}")
print(f"    Sesudah : {df['step_stopword'].iloc[0]}")


[4] Stopword Removal...
    Membuang kata umum tidak bermakna
    Catatan: kata kasar TIDAK dihapus agar tetap jadi fitur model

    Contoh:
    Sebelum : ['dasar', 'sialan']
    Sesudah : ['dasar', 'sialan']


In [41]:
print("\n[5] Stemming...")
print("    Mengubah kata ke bentuk dasar (Algoritma Nazief-Adriani)")
print("    Token khusus & kata kasar SKIP stemming")
print("    Proses ini memerlukan waktu, harap tunggu...")
 
def stemming(tokens):
    return [
        t if (t.isupper() or t in KATA_KASAR)
        else stemmer.stem(t)
        for t in tokens
    ]
 
start = time.time()
df['step_stemming'] = df['step_stopword'].apply(stemming)
df['teks_bersih']   = df['step_stemming'].apply(
    lambda t: ' '.join(t) if t else 'PESAN_KOSONG'
)
elapsed = time.time() - start
print(f"    Selesai dalam {elapsed:.1f} detik")
 
# Tampilkan ringkasan per tahap untuk contoh Berisiko & Tidak Berisiko
print("\n    ── Ringkasan 5 Tahap Preprocessing ──")
for lbl in ['Berisiko', 'Tidak Berisiko']:
    i = df[df['label'] == lbl].index[0]
    print(f"\n    [{lbl}]")
    print(f"    Asli       : {df['pesan'][i]}")
    print(f"    CaseFold   : {df['step_casefolding'][i]}")
    print(f"    Tokenisasi : {df['step_tokenisasi'][i]}")
    print(f"    Stopword   : {df['step_stopword'][i]}")
    print(f"    Stemming   : {df['step_stemming'][i]}")
    print(f"    Teks Final : {df['teks_bersih'][i]}")


[5] Stemming...
    Mengubah kata ke bentuk dasar (Algoritma Nazief-Adriani)
    Token khusus & kata kasar SKIP stemming
    Proses ini memerlukan waktu, harap tunggu...
    Selesai dalam 39.4 detik

    ── Ringkasan 5 Tahap Preprocessing ──

    [Berisiko]
    Asli       : dasar sialan
    CaseFold   : dasar sialan
    Tokenisasi : ['dasar', 'sialan']
    Stopword   : ['dasar', 'sialan']
    Stemming   : ['dasar', 'sialan']
    Teks Final : dasar sialan

    [Tidak Berisiko]
    Asli       : Transfer berhasil. Rp45.000 telah dikirim ke rekening BRI atas nama Rina.
    CaseFold   : transfer berhasil. rp45.000 telah dikirim ke rekening bri atas nama rina.
    Tokenisasi : ['transfer', 'berhasil', '_', 'telah', 'dikirim', 'ke', 'rekening', 'bri', 'atas', 'nama', 'rina']
    Stopword   : ['transfer', 'berhasil', 'dikirim', 'rekening', 'bri', 'atas', 'nama', 'rina']
    Stemming   : ['transfer', 'hasil', 'kirim', 'rekening', 'bri', 'atas', 'nama', 'rina']
    Teks Final : transfer hasil k

In [42]:
print("\n[6] Split Data Training dan Testing (80:20)...")


[6] Split Data Training dan Testing (80:20)...


In [43]:
label_map = {'Berisiko': 1, 'Tidak Berisiko': 0}
X = df['teks_bersih']
y = df['label'].map(label_map)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
print(f"    Data Training : {len(X_train)} data (80%)")
print(f"    Data Testing  : {len(X_test)} data (20%)")
print(f"    Label encoding: Berisiko=1, Tidak Berisiko=0")

    Data Training : 988 data (80%)
    Data Testing  : 247 data (20%)
    Label encoding: Berisiko=1, Tidak Berisiko=0


In [44]:
print("\n[7] Ekstraksi Fitur — TF-IDF...")
print("    Mengubah teks menjadi representasi angka (vektor bobot)")
 
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigram + bigram
    max_features=5000,    # 5000 fitur terpenting
    sublinear_tf=True,    # log TF
    min_df=1              # masukkan semua kata
)
 
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
 
feature_names = tfidf.get_feature_names_out()
mean_tfidf    = X_train_tfidf.mean(axis=0).A1
top10_idx     = mean_tfidf.argsort()[::-1][:10]
 
print(f"\n    Dimensi matriks training  : {X_train_tfidf.shape}")
print(f"    Dimensi matriks testing   : {X_test_tfidf.shape}")
print(f"    Jumlah fitur (vocabulary) : {len(tfidf.vocabulary_)}")
print(f"    Top 10 fitur TF-IDF       : {[feature_names[i] for i in top10_idx]}")


[7] Ekstraksi Fitur — TF-IDF...
    Mengubah teks menjadi representasi angka (vektor bobot)

    Dimensi matriks training  : (988, 3620)
    Dimensi matriks testing   : (247, 3620)
    Jumlah fitur (vocabulary) : 3620
    Top 10 fitur TF-IDF       : ['lo', 'banget', 'hasil', 'gue', 'akun', 'bulan', 'dasar', 'pukul', 'bayar', 'jam']


In [45]:
print("\n[8] Training Model Multinomial Naïve Bayes...")
print("    alpha = 0.3 (Laplace smoothing)")
 
nb_model = MultinomialNB(alpha=0.3)
nb_model.fit(X_train_tfidf, y_train)
print("    Model selesai dilatih")


[8] Training Model Multinomial Naïve Bayes...
    alpha = 0.3 (Laplace smoothing)
    Model selesai dilatih


In [46]:
print("\n[9] Evaluasi Model...")
 
y_pred = nb_model.predict(X_test_tfidf)
 
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
cm   = confusion_matrix(y_test, y_pred)
 
TP = cm[1][1]
FN = cm[1][0]
FP = cm[0][1]
TN = cm[0][0]


[9] Evaluasi Model...


In [47]:
print(f"\n    Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"    Precision : {prec:.4f}  ({prec*100:.2f}%)")
print(f"    Recall    : {rec:.4f}  ({rec*100:.2f}%)")
print(f"    F1-Score  : {f1:.4f}  ({f1*100:.2f}%)")
print()
print("    Confusion Matrix:")
print(f"    {'':26s}  Pred: Berisiko   Pred: Tidak Berisiko")
print(f"    {'Aktual: Berisiko':26s}  TP={TP:8d}       FN={FN:8d}")
print(f"    {'Aktual: Tidak Berisiko':26s}  FP={FP:8d}       TN={TN:8d}")
print()
print("    Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Berisiko', 'Berisiko']))


    Accuracy  : 0.9717  (97.17%)
    Precision : 0.9693  (96.93%)
    Recall    : 0.9875  (98.75%)
    F1-Score  : 0.9783  (97.83%)

    Confusion Matrix:
                                Pred: Berisiko   Pred: Tidak Berisiko
    Aktual: Berisiko            TP=     158       FN=       2
    Aktual: Tidak Berisiko      FP=       5       TN=      82

    Classification Report:
                precision    recall  f1-score   support

Tidak Berisiko       0.98      0.94      0.96        87
      Berisiko       0.97      0.99      0.98       160

      accuracy                           0.97       247
     macro avg       0.97      0.97      0.97       247
  weighted avg       0.97      0.97      0.97       247



In [48]:
print("[10] Menyimpan model ke file joblib...")
 
os.makedirs('model', exist_ok=True)
output_path = 'model/model_naive_bayes.joblib'
 
# PENTING: Simpan TANPA fungsi/lambda agar aman di-load di manapun
joblib.dump({
    'model':        nb_model,       # MultinomialNB
    'vectorizer':   tfidf,          # TfidfVectorizer
    'stopwords':    list(STOPWORDS),
    'kata_kasar':   list(KATA_KASAR),
    'pola_url':     POLA_URL,
    'label_map':    label_map,      # {'Berisiko':1, 'Tidak Berisiko':0}
    'metadata': {
        'total_data':   len(df),
        'train_size':   len(X_train),
        'test_size':    len(X_test),
        'accuracy':     round(acc,  4),
        'precision':    round(prec, 4),
        'recall':       round(rec,  4),
        'f1_score':     round(f1,   4),
        'confusion_matrix': cm.tolist(),
        'label_encoding':   '1=Berisiko, 0=Tidak Berisiko',
        'kategori':         list(df['kategori'].unique()),
        'preprocessing_steps': [
            '1. Case Folding (lowercase)',
            '2. Tokenisasi + penandaan URL_CURIGA/NOMOR_HP_ASING/KODE_OTP/NOMINAL_UANG',
            '3. Stopword Removal (Sastrawi + custom, kata kasar dijaga)',
            '4. Stemming Nazief-Adriani via Sastrawi (token khusus & kata kasar skip)',
            '5. Ekstraksi Fitur TF-IDF (1,2)-gram max 5000 fitur',
        ],
        'catatan': (
            'Sistem menggunakan 2 lapis deteksi: '
            '(1) Blacklist kata kasar → langsung Berisiko, '
            '(2) Model Naïve Bayes untuk pola ancaman lainnya. '
            'Pesan < 4 karakter langsung Tidak Berisiko (percakapan biasa).'
        )
    }
}, output_path)
 
size_kb = os.path.getsize(output_path) / 1024
print(f"    File    : {output_path}")
print(f"    Ukuran  : {size_kb:.1f} KB")

[10] Menyimpan model ke file joblib...
    File    : model/model_naive_bayes.joblib
    Ukuran  : 272.7 KB


In [53]:
print("\n[11] Uji Prediksi dengan data baru...")
 
# ── Konstanta tambahan ────────────────────────────────────────────
# Kata yang ada di blacklist tapi bisa jadi konteks berbeda (perlu cek konteks)
KATA_AMBIGUOUS = {
    'porno', 'bugil', 'telanjang', 'mesum',
}

def _is_kemungkinan_nama(pesan_asli: str) -> bool:
    """Cek apakah pesan hanya berisi nama orang (tiap kata diawali kapital)."""
    import re as _re
    words = pesan_asli.strip().split()
    if not words or len(words) > 5:
        return False
    all_cap = all(w[0].isupper() for w in words if w)
    no_special = bool(_re.match(r'^[A-Za-z\s]+$', pesan_asli.strip()))
    return all_cap and no_special

# ── Fungsi prediksi final (diperbaiki) ───────────────────────────
def prediksi(pesan):
    pesan = str(pesan).strip()

    # Rule 1: terlalu pendek
    if len(pesan) < 4:
        return {'label': 'Tidak Berisiko', 'keyakinan': 99.0, 'alasan': 'Terlalu pendek'}

    # Rule 2a: kemungkinan nama orang → skip blacklist
    if _is_kemungkinan_nama(pesan):
        return {'label': 'Tidak Berisiko', 'keyakinan': 95.0, 'alasan': 'Kemungkinan nama orang'}

    # Rule 2b: blacklist kata kasar KERAS (tidak termasuk kata ambiguous)
    t      = re.sub(r'[^a-z\s]', ' ', pesan.lower())
    tokens = set(t.split())
    KATA_KASAR_KERAS = KATA_KASAR - KATA_AMBIGUOUS
    if tokens & KATA_KASAR_KERAS:
        found = tokens & KATA_KASAR_KERAS
        return {'label': 'Berisiko', 'keyakinan': 99.0,
                'alasan': f'Kata kasar terdeteksi: {", ".join(sorted(found))}'}

    # Rule 2c: kata ambiguous hanya Berisiko jika ada konteks negatif
    if tokens & KATA_AMBIGUOUS:
        KONTEKS_NEGATIF = {
            'yuk','ayo','mau','sini','coba','nonton','lihat','download',
            'kirim','bagi','share','klik','link','join','masuk'
        }
        if tokens & KONTEKS_NEGATIF:
            found = tokens & KATA_AMBIGUOUS
            return {'label': 'Berisiko', 'keyakinan': 92.0,
                    'alasan': f'Konten tidak pantas: {", ".join(sorted(found))}'}

    # Rule 2d: URL mencurigakan (sebelum preprocessing)
    if re.search(POLA_URL, pesan.lower()):
        return {
            'label': 'Berisiko',
            'keyakinan': 92.0,
            'alasan': 'URL mencurigakan terdeteksi'
        }

    # Preprocessing
    t = pesan.lower().strip()
    t = re.sub(POLA_URL, 'URL_CURIGA', t)
    t = re.sub(r'\b0\d[\d\-]{8,12}\b', 'NOMOR_HP_ASING', t)
    t = re.sub(r'\b\d{5,8}\b', 'KODE_OTP', t)
    t = re.sub(r'rp[\s]?\d+[,.]?\d*\s*(juta|ribu|rb)?', 'NOMINAL_UANG', t)
    t = re.sub(r'\b\d+\b', '', t)
    t = re.sub(r'[^a-z_\s]', ' ', t)
    tks = [x for x in t.split() if len(x) > 0]
    tks = [x for x in tks if x.isupper() or x in KATA_KASAR or (x not in STOPWORDS and len(x) > 1)]
    tks = [x if (x.isupper() or x in KATA_KASAR) else stemmer.stem(x) for x in tks]
    bersih = ' '.join(tks) if tks else 'PESAN_KOSONG'

    # Rule 3: token hampir kosong → percakapan biasa
    if len(tks) < 2:
        return {'label': 'Tidak Berisiko', 'keyakinan': 90.0, 'alasan': 'Token tidak bermakna'}

    # Model NB dengan threshold minimum 65% keyakinan untuk label Berisiko
    vec  = tfidf.transform([bersih])
    prob = nb_model.predict_proba(vec)[0]
    label = 'Berisiko' if prob[1] >= 0.65 else 'Tidak Berisiko'
    return {'label': label, 'keyakinan': round(max(prob)*100, 2), 'alasan': 'Model NB'}


# ── Test cases ────────────────────────────────────────────────────
test_cases = [
    # Percakapan sehari-hari → Tidak Berisiko
    
    # Phishing & ancaman → Berisiko
    ("Berisiko", "bit://iy"),  # URL mencurigakan
    # ("Berisiko", "Selamat menang Rp50 juta. Klik link ini untuk klaim"),
    # ("Berisiko", "Berikan kode OTP 819234 kepada saya untuk verifikasi"),
    # ("Berisiko", "Download APK WhatsApp terbaru di shorturl.at/wa-update"),
    # ("Berisiko", "Bisnis online profit 30% dijamin WA 081234567890"),
    
    #  ("Berisiko", "Kamu benar-benar tolol, kenapa nggak pernah paham sama instruksi?"),
    # ("Berisiko", "Hati-hati ngomong, kalau nggak mau jadi bajingan di sini!"),
    # ("Berisiko", "Sumpah, kamu kayak anjing aja sih, nggak pernah nyelesain tugas dengan baik."),
    # ("Berisiko", "Mending jangan bangsat ngomong kayak gitu, aku cuma ngikutin perintah."),
    # ("Berisiko", "Serius deh, kok kamu bodoh banget sih, nggak ngerti yang jelas-jelas dikasih tahu?"),

    # Percakapan dengan Kata-Kata Tidak Kasar (Netral/Positif) → Tidak Berisiko
    # ("Tidak Berisiko", "Kamu itu cerdas, bisa menjelaskan hal yang susah jadi gampang dipahami."),
    # ("Tidak Berisiko", "Makasih, aku akan berusaha lebih baik dan selalu bijaksana dalam setiap keputusan."),
    # ("Tidak Berisiko", "Saya percaya kamu bisa melakukannya dengan hasil yang hebat."),
    # ("Tidak Berisiko", "Terima kasih atas dukungannya, saya akan selalu ramah dan siap membantu."),
    # ("Tidak Berisiko", "Tetap sabar dan jangan cepat menyerah, ya. Semua usaha akan membuahkan hasil."),

    # # Percakapan dengan Kata-Kata yang Berindikasi Penipuan → Berisiko
    # ("Berisiko", "Daftar sekarang, kamu bisa dapat uang banyak dalam waktu singkat, cuma klik link ini!"),
    # ("Berisiko", "Gak perlu modal, kamu cuma perlu klik di sini dan segera klaim bonus!"),
    # ("Berisiko", "Jangan sampai ketinggalan! Penawaran terbatas, cuma berlaku hari ini!"),
    # ("Berisiko", "Cuma dengan sedikit usaha, kamu bisa cepat kaya tanpa risiko apapun!"),
    # ("Berisiko", "Jangan ragu, ini adalah jaminan keuntungan tanpa modal, langsung dapat uang!"),
    
    ("Tidak Berisiko", "Dimas Kurniawan"),
    ("Tidak Berisiko", "Belum nih, lagi mikirin kerjaan."),
    ("Tidak Berisiko", "Ya ampun, makan dulu dong, biar nggak pusing."),
    ("Tidak Berisiko", "Iya, nanti deh. Masih banyak yang harus dikerjain."),
    ("Tidak Berisiko", "Wah, jangan terlalu dipaksain, nanti sakit."),
    ("Tidak Berisiko", "Iya juga sih, sebentar lagi mau makan."),

    # Percakapan Formal dan Sopan
    ("Tidak Berisiko", "Selamat pagi, apakah Anda sudah menerima email saya kemarin?"),
    ("Tidak Berisiko", "Selamat pagi, saya sudah menerimanya. Terima kasih telah mengirimkan dokumen tersebut."),
    ("Tidak Berisiko", "Sama-sama, apakah ada hal lain yang perlu saya bantu terkait dokumen tersebut?"),
    ("Tidak Berisiko", "Saat ini saya sedang mempelajari isinya, dan akan menghubungi Anda jika ada pertanyaan."),
    ("Tidak Berisiko", "Baik, saya akan menunggu kabar dari Anda."),

    # Percakapan Kasar (Untuk Pengujian Kode Kasar)
    ("Berisiko", "Kenapa sih nggak bisa ngerti-ngerti juga instruksi yang udah jelas?"),
    ("Berisiko", "Lo juga ngomongnya gampang, emang gampang?"),
    ("Berisiko", "Ya iyalah, lo kenapa sih susah banget buat paham?"),
    ("Berisiko", "Mending lo diem aja deh, nggak usah nyalahin gue!"),
    ("Berisiko", "Terserah lo, gue sih udah cape."),

    # Percakapan Netral (Tanpa Kasar)
    ("Tidak Berisiko", "Kenapa sih nggak bisa ngerti instruksinya?"),
    ("Tidak Berisiko", "Maaf, saya agak kesulitan dengan penjelasannya. Bisa dijelaskan lebih jelas?"),
    ("Tidak Berisiko", "Oh, oke. Coba gue jelasin lagi, ini instruksinya..."),
    ("Tidak Berisiko", "Terima kasih, sekarang saya lebih paham."),
    
    # Percakapan dengan Kata-Kata yang Berindikasi Penipuan → Berisiko
    ("Berisiko", "Daftar sekarang, kamu bisa dapat uang banyak dalam waktu singkat, cuma klik link ini!"),
    ("Berisiko", "Gak perlu modal, kamu cuma perlu klik di sini dan segera klaim bonus!"),
    ("Berisiko", "Jangan sampai ketinggalan! Penawaran terbatas, cuma berlaku hari ini!"),
    ("Berisiko", "Cuma dengan sedikit usaha, kamu bisa cepat kaya tanpa risiko apapun!"),
    ("Berisiko", "Jangan ragu, ini adalah jaminan keuntungan tanpa modal, langsung dapat uang!"),
]

print(f"\n    {'Ekspektasi':16s}  {'Hasil':16s}  {'Yakin':7s}  Sts  Alasan / Pesan")
print("    " + "-" * 90)

benar = 0
for ekspektasi, pesan in test_cases:
    hasil  = prediksi(pesan)
    status = "✓ OK" if hasil['label'] == ekspektasi else "✗ SALAH"
    if hasil['label'] == ekspektasi:
        benar += 1
    print(f"    {ekspektasi:16s}  {hasil['label']:16s}  {hasil['keyakinan']:5.1f}%  "
          f"{status}  [{hasil['alasan']}] {pesan[:40]}")

print(f"\n    Hasil : {benar}/{len(test_cases)} benar ({benar/len(test_cases)*100:.0f}%)")
print("\n" + "=" * 60)
print("  SELESAI — model/model_naive_bayes.joblib siap digunakan")
print("=" * 60)



[11] Uji Prediksi dengan data baru...

    Ekspektasi        Hasil             Yakin    Sts  Alasan / Pesan
    ------------------------------------------------------------------------------------------
    Berisiko          Tidak Berisiko     64.8%  ✗ SALAH  [Model NB] bit://iy
    Tidak Berisiko    Tidak Berisiko     95.0%  ✓ OK  [Kemungkinan nama orang] Dimas Kurniawan
    Tidak Berisiko    Tidak Berisiko     58.4%  ✓ OK  [Model NB] Belum nih, lagi mikirin kerjaan.
    Tidak Berisiko    Tidak Berisiko     71.9%  ✓ OK  [Model NB] Ya ampun, makan dulu dong, biar nggak pu
    Tidak Berisiko    Tidak Berisiko     77.0%  ✓ OK  [Model NB] Iya, nanti deh. Masih banyak yang harus 
    Tidak Berisiko    Tidak Berisiko     59.8%  ✓ OK  [Model NB] Wah, jangan terlalu dipaksain, nanti sak
    Tidak Berisiko    Tidak Berisiko     83.8%  ✓ OK  [Model NB] Iya juga sih, sebentar lagi mau makan.
    Tidak Berisiko    Tidak Berisiko     92.0%  ✓ OK  [Model NB] Selamat pagi, apakah Anda sudah menerim